# Search Application
We are going to build a search application for our education startup. Our startup is a non-profit organization that provides free education to students in developing countries. Our startup has a large number of YouTube videos that students can use to learn about AI. Our startup wants to build a search application that allows students to search for a YouTube video by typing a question.

In [58]:
# Import Libraries
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Configure DeepSeek as used LLM
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)
deployment = "text-embedding-ada-002"

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"

Next, we are going to load the embedding index into a Pandas DataFrame. The embedding index is stored in a JSON file. The embedding index contains the embeddings for each of the YouTube transcripts up until Oct 2023.

In [59]:
# Define function to create Pandas DataFrame
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

Next, we are going to create a function called `get_videos` that will search the embedding index for the query. The function will return the top 5 videos that are most similar to the query. The function works as follows: 
1. First, a copy of embedding index is created.
2. Second, the embedding for the query is calculated using the OpenAI embedding API.
3. Then a new column is created in the embedding index called `similarity`. The `similarity` column contains the cosine similarity between the query embedding and the embedding for each video segment.
4. Next, the embedding index is filtered by the `similarity` column. The embedding index is filtered to only include videos that have a cosine similarity greater than or equal to 0.75.
5. Finally, the embedding index is sorted by the `similarity` column and the top 5 videos are returned.

In [60]:
# Define function to calculate cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Define function to search embedding index
def get_videos(query: str, dataset: pd.core.frame.DataFrame, rows: int) -> pd.core.frame.DataFrame:
    # Create a copy of the dataset
    video_vectors = dataset.copy()

    print(deployment)
    print(query)
    query_embeddings = client.embeddings.create(model=deployment, input=[query]).data[0].embedding

    # Create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["ada_v2"].apply(
        lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
    )

    # Filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(rows)
    return video_vectors.head(rows)

# Define function to print the results
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_youtube_url(video_id: str, seconds: int) -> str:
        """Convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}': ")
    for _, row in videos.iterrows():
        youtube_url = _gen_youtube_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

You will be prompted to enter a query. Enter a query and press enter. The application will return a list of videos that are relevant to the query. The application will also return a link to the place in the video where the answer to the question is located.

In [61]:
pd_vectors = load_dataset(DATASET_NAME)

# Get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)

text-embedding-ada-002
what is azure machine learning?

Videos similar to 'what is azure machine learning?': 
 - Azure Machine Learning Studio
   Summary: Stephanie Lindsay, a Program Manager at Azure Machine Learning, introduces the new Azure Machine Learning...
   YouTube: https://youtu.be/JNa4VV0d8T0?t=0
   Similarity: 0.8707596143957488
   Speakers: Stephanie Lindsay
 - Time Series Forecasting with Azure Machine Learning
   Summary: In this video, the speaker discusses time series forecasting with Azure Machine Learning. They demonstrate...
   YouTube: https://youtu.be/mGr_c2UnOUI?t=7
   Similarity: 0.8674860086761169
   Speakers: Hi, everyone. In this video, we are going to talk about time series forecasting with Azure Machine Learning.
 - What’s new with Azure Machine Learning
   Summary: Azure Machine Learning is a powerful tool for training and deploying machine learning models. It...
   YouTube: https://youtu.be/YlWCeY_CWEg?t=914
   Similarity: 0.8668445636944175
   Speakers: 